# Structural Determinants of Landfill Dependency in Emerging & Transition European Economies
## Final panel-econometric analysis (revised per reviewer reports)

Balanced-as-possible country–year panel of **9 economies, 2015–2023**. Dependent variable:
the **landfill rate** (share of municipal waste landfilled, 0–1). This notebook implements the
revised design and produces every table/figure for the manuscript. Run **Runtime → Run all**
(seed = 42); tables are written to `outputs/`.

**What changed vs. the original submission**
1. Clustering is **descriptive only** and rebuilt on **predictor-side variables only** (outcome and
   outcome-derived variables excluded); regime dummies are **not** regressors. *(R1)*
2. `treatment_intensity` is **dropped** (under Eurostat `env_wasmun`, "treated" includes
   landfilling, so it is mechanically tied to the outcome). *(R1)*
3. Main estimators are **fixed effects** and a **Correlated Random Effects (Mundlak)** model that
   separates **within-country** from **between-country** variation; pooled OLS is a benchmark. *(R2)*
4. Inference uses the **wild cluster bootstrap (Webb weights)** with confidence intervals. *(R3)*
5. Digitalization claims are narrowed: within vs. between effects, CI in substantive units, an
   **equivalence (TOST)** test, and a **mediation check**; no untested "mediation" claim. *(R4)*
6. The **EKC** reading is tested with a quadratic GDP term + turning point; a **fractional-response**
   robustness check is added for the [0,1] outcome. *(R5)*
7. **Sample construction** documented (missing obs, interpolations) with leave-one-country-out and
   exclude-interpolated robustness. *(R6)*
8. The Tirana forecasting exercise has been **removed** (it measured a single municipality vs. the
   national panel — not comparable). *(R7 / R1)*


## 0. Setup

In [ ]:
!pip -q install linearmodels pingouin statsmodels scikit-learn
print("ok")

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.graphics.gofplots import qqplot
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS

RANDOM_STATE=42
OUTPUT_DIR="outputs"; os.makedirs(OUTPUT_DIR, exist_ok=True)
BASE_URL=""                       # set to GitHub raw path, or "" for local/upload
PANEL_CSV="data/processed/panel_landfill_2015_2023.csv"
pd.set_option("display.width",200); pd.set_option("display.max_columns",60)
print("imports ok")

In [ ]:
def load_csv(name):
    src=(BASE_URL+name) if BASE_URL else name
    try: return pd.read_csv(src)
    except (FileNotFoundError,OSError):
        from google.colab import files
        print(f"Upload {name}:"); up=files.upload()
        return pd.read_csv(name if name in up else list(up.keys())[0])
raw=load_csv(PANEL_CSV); print("shape:",raw.shape)

## 1. Data preparation and sample documentation *(R6)*

Nine countries, 2015–2023. The panel is **not perfectly balanced**: Latvia is missing 2023
(8 obs), so N = 80 rather than 81. Two isolated values were linearly interpolated
(Montenegro 2017; Serbia 2020). North Macedonia and Bosnia & Herzegovina were excluded for
extensive gaps. `treatment_intensity` is kept in the data for transparency but **excluded from
all models**.

In [ ]:
COLMAP={
 "Shteti":"country","Viti":"year",
 "norma_e_depozitimit_ne_landfill":"landfill_rate",
 "norma_e_trajtimit_jo_ne_landfill":"nonlandfill_rate",
 "mbetje_per_fryme":"waste_per_capita","intensiteti_trajtimit":"treatment_intensity",
 "PBB_per_fryme":"gdp_per_capita",
 "popullesia_urbane_perqindja_mbi_popullesine_totale":"urban_pop",
 "papunesia_perqindja_totale_fuqise_punetore":"unemployment",
 "perdorimi_internetit_nga_individe":"internet"}
df=raw.rename(columns=COLMAP)
for c in ["landfill_rate","waste_per_capita","gdp_per_capita","urban_pop","unemployment",
          "internet","treatment_intensity","nonlandfill_rate"]:
    if c in df.columns: df[c]=pd.to_numeric(df[c],errors="coerce")
df=df.sort_values(["country","year"]).reset_index(drop=True)
df["trend"]=df["year"]-df["year"].min()

COV=["waste_per_capita","gdp_per_capita","urban_pop","unemployment","internet"]  # model covariates
YVAR="landfill_rate"
model_df=df[["country","year",YVAR,"trend"]+COV].dropna().reset_index(drop=True)

struct=df.groupby("country")["year"].agg(["count","min","max"])
print("Observations per country:"); print(struct.to_string())
print(f"\nN={len(model_df)} | countries={model_df['country'].nunique()} | "
      f"balanced would be {model_df['country'].nunique()*df['year'].nunique()} (Latvia missing 2023)")

### 1.1 Descriptive statistics *(Table)*

In [ ]:
labels={"landfill_rate":"Landfill rate (0-1)","waste_per_capita":"Waste per capita (kg/inh.)",
 "gdp_per_capita":"GDP per capita (index)","internet":"Internet use (%)",
 "urban_pop":"Urban population (%)","unemployment":"Unemployment (%)"}
dv=list(labels)
desc=pd.DataFrame({"Variable":[labels[v] for v in dv],"N":[df[v].notna().sum() for v in dv],
 "Mean":[df[v].mean() for v in dv],"SD":[df[v].std() for v in dv],
 "Min":[df[v].min() for v in dv],"Max":[df[v].max() for v in dv],
 "Median":[df[v].median() for v in dv]}).round(3)
desc.to_csv(f"{OUTPUT_DIR}/T01_descriptives.csv",index=False); desc

## 2. Descriptive typology — predictor-side clustering *(R1)*

Rebuilt on **predictor-side variables only**; the outcome and outcome-derived variables are
excluded. Regimes are a **descriptive typology only** and are **not** used as regressors.

In [ ]:
pred=["waste_per_capita","gdp_per_capita","unemployment","urban_pop","internet"]
Xz=StandardScaler().fit_transform(model_df[pred])
sel=pd.DataFrame([{"K":k,"inertia":round(KMeans(n_clusters=k,n_init=25,random_state=RANDOM_STATE).fit(Xz).inertia_,2),
    "silhouette":round(silhouette_score(Xz,KMeans(n_clusters=k,n_init=25,random_state=RANDOM_STATE).fit_predict(Xz)),3)}
    for k in range(2,7)])
sel.to_csv(f"{OUTPUT_DIR}/T02_cluster_selection.csv",index=False)
print("Cluster-selection (KMeans n_init=25, seed=42):"); print(sel.to_string(index=False))
km=KMeans(n_clusters=3,n_init=25,random_state=RANDOM_STATE).fit(Xz)
model_df["regime"]=km.labels_
if "cluster" in raw.columns:
    old=raw.rename(columns={"Shteti":"country","Viti":"year"})[["country","year","cluster"]]
    mm=model_df.merge(old,on=["country","year"],how="left")
    print(f"\nARI(new predictor-side vs old outcome-based) = "
          f"{adjusted_rand_score(mm['cluster'].astype(int),mm['regime'].astype(int)):.3f}")
prof=model_df.groupby("regime").agg(n=("landfill_rate","size"),landfill=("landfill_rate","mean"),
    waste=("waste_per_capita","mean"),gdp=("gdp_per_capita","mean"),internet=("internet","mean")).round(3)
prof.to_csv(f"{OUTPUT_DIR}/T03_regime_profiles.csv")
print("\nDescriptive profile by regime:"); print(prof.to_string())
print("NOTE: regimes are descriptive only; NOT entered into any regression.")

## 3. Panel models: Pooled OLS, FE, RE *(R2)*

Covariates: waste per capita, GDP per capita, urban population, unemployment, internet.
`treatment_intensity` and regime dummies are excluded. SEs clustered by country.

In [ ]:
pdata=model_df.set_index(["country","year"]); y=pdata[YVAR]
def summ(res,name):
    print(f"\n=== {name} ===  R2={res.rsquared:.3f}  within={getattr(res,'rsquared_within',np.nan):.3f}"
          f"  between={getattr(res,'rsquared_between',np.nan):.3f}  N={int(res.nobs)}")
    t=pd.DataFrame({"coef":res.params,"se":res.std_errors,"t":res.tstats,"p":res.pvalues,
        "ci_low":res.conf_int()["lower"],"ci_high":res.conf_int()["upper"]}).round(4)
    print(t.to_string()); return t
pooled=PooledOLS(y,sm.add_constant(pdata[COV])).fit(cov_type="clustered",cluster_entity=True)
cfe=PanelOLS(y,sm.add_constant(pdata[COV]),entity_effects=True).fit(cov_type="clustered",cluster_entity=True)
twfe=PanelOLS(y,sm.add_constant(pdata[COV]),entity_effects=True,time_effects=True).fit(cov_type="clustered",cluster_entity=True)
re=RandomEffects(y,sm.add_constant(pdata[COV])).fit(cov_type="clustered",cluster_entity=True)
t1=summ(pooled,"(1) Pooled OLS [benchmark]"); t2=summ(cfe,"(2) Country FE")
t3=summ(twfe,"(3) Two-way FE"); t4=summ(re,"(4) Random Effects")
for nm,t in [("pooled",t1),("countryFE",t2),("twowayFE",t3),("RE",t4)]:
    t.to_csv(f"{OUTPUT_DIR}/T04_model_{nm}.csv")
# variance decomposition + AIC/BIC
lsdv=smf.ols("landfill_rate ~ "+" + ".join(COV)+" + C(country)",data=model_df).fit()
bc=model_df.groupby("country")["landfill_rate"].mean().var(ddof=0)/model_df["landfill_rate"].var(ddof=0)
print(f"\nBetween-country share of landfill variance = {bc:.3f} (structural, cross-country)")
print(f"LSDV: R2={lsdv.rsquared:.3f}  AIC={lsdv.aic:.1f}  BIC={lsdv.bic:.1f}")

## 4. Correlated Random Effects (Mundlak) — main specification *(R1, R2)*

The CRE model adds the **country means** of each time-varying regressor to a random-effects
specification. This yields, in a single model: (i) the **within-country** coefficients (identical
to FE), (ii) the **between-country** coefficients (the `m_` means), and (iii) the **Mundlak test**
(joint significance of the means), which formally justifies FE over RE. This directly delivers the
between/within decomposition requested in review.

In [ ]:
cre=model_df.copy()
for v in COV: cre["m_"+v]=cre.groupby("country")[v].transform("mean")
means=["m_"+v for v in COV]
X=sm.add_constant(cre[COV+means].astype(float))
mcre=sm.OLS(cre[YVAR].astype(float),X).fit(cov_type="cluster",cov_kwds={"groups":cre["country"]})

win=pd.DataFrame({"component":"within","coef":mcre.params[COV].round(5),"p":mcre.pvalues[COV].round(3),
    "ci_low":mcre.conf_int().loc[COV,0].round(5),"ci_high":mcre.conf_int().loc[COV,1].round(5)})
bet=pd.DataFrame({"component":"between","coef":mcre.params[means].values.round(5),"p":mcre.pvalues[means].values.round(3),
    "ci_low":mcre.conf_int().loc[means,0].values.round(5),"ci_high":mcre.conf_int().loc[means,1].values.round(5)},
    index=COV)
cre_tab=pd.concat([win,bet]); cre_tab.to_csv(f"{OUTPUT_DIR}/T05_CRE_mundlak.csv")
print("CRE / MUNDLAK (cluster-robust)"); print("\nWITHIN (= FE estimates):"); print(win.to_string())
print("\nBETWEEN (country means):"); print(bet.to_string())

# Mundlak test: joint significance of the means -> FE vs RE
R=np.zeros((len(means),len(mcre.params)))
for i,mm in enumerate(means): R[i,list(mcre.params.index).index(mm)]=1
w=mcre.f_test(R)
print(f"\nMUNDLAK TEST (H0: between-means = 0): F={float(w.fvalue):.2f}, p={float(w.pvalue):.4f} -> "
      f"{'FE preferred over RE' if w.pvalue<0.05 else 'RE adequate'}")

## 5. Small-cluster inference: wild cluster bootstrap (Webb) *(R3)*

With G = 9, conventional cluster-robust *t*-tests are unreliable. We report a restricted wild
cluster bootstrap-*t* with Webb 6-point weights (recommended for G < 12), for each covariate in
the FE specification, with the associated confidence sets.

In [ ]:
def wcr(data,yname,xnames,cluster,tv,B=1999,seed=RANDOM_STATE):
    rng=np.random.default_rng(seed)
    X=sm.add_constant(data[xnames].astype(float)); yv=data[yname].astype(float)
    g=data[cluster].astype(str).values
    full=sm.OLS(yv,X).fit(cov_type="cluster",cov_kwds={"groups":g})
    t0=full.params[tv]/full.bse[tv]
    restr=sm.OLS(yv,X.drop(columns=[tv])).fit(); yhat=restr.fittedvalues.values; u=restr.resid.values
    cl=np.unique(g); webb=np.array([-np.sqrt(1.5),-1,-np.sqrt(.5),np.sqrt(.5),1,np.sqrt(1.5)]); ts=np.empty(B)
    for b in range(B):
        w=dict(zip(cl,rng.choice(webb,size=len(cl)))); wv=np.array([w[gi] for gi in g])
        fb=sm.OLS(yhat+wv*u,X).fit(cov_type="cluster",cov_kwds={"groups":g}); ts[b]=fb.params[tv]/fb.bse[tv]
    return t0,np.mean(np.abs(ts)>=np.abs(t0)),full.params[tv],full.conf_int().loc[tv].values
cd=pd.get_dummies(model_df["country"],prefix="c",drop_first=True).astype(float)
bdf=pd.concat([model_df,cd],axis=1); xn=COV+list(cd.columns)
rows=[]
for v in COV:
    t0,pw,beta,ci=wcr(bdf,YVAR,xn,"country",v)
    rows.append({"Variable":v,"beta":round(beta,5),"CI_low":round(ci[0],5),
                 "CI_high":round(ci[1],5),"wild_p_Webb":round(pw,3)})
wild=pd.DataFrame(rows)
wild.to_csv(f"{OUTPUT_DIR}/T06_wildbootstrap.csv",index=False)
print("Wild cluster bootstrap (Webb, B=1999), Country FE:"); print(wild.to_string(index=False))

## 6. Panel diagnostics *(R2)*

In [ ]:
def pesaran_cd(data,yname,xnames,ent,tm):
    d=data.copy(); d["res"]=sm.OLS(d[yname].astype(float),sm.add_constant(d[xnames].astype(float))).fit().resid
    piv=d.pivot(index=tm,columns=ent,values="res"); e=piv.columns; N=len(e); T=len(piv); s=0;c=0
    for i in range(N):
        for j in range(i+1,N):
            a,b=piv[e[i]],piv[e[j]]; m=a.notna()&b.notna()
            if m.sum()>3: s+=np.corrcoef(a[m],b[m])[0,1]; c+=1
    CD=np.sqrt(2*T/(N*(N-1)))*s; return CD,2*(1-stats.norm.cdf(abs(CD)))
cs,cp=pesaran_cd(model_df,YVAR,COV,"country","year")
print(f"Pesaran CD: stat={cs:.3f}, p={cp:.4f} -> {'CD present' if cp<0.05 else 'no cross-sectional dependence'}")
r=np.asarray(cfe.resids).ravel(); jb=stats.jarque_bera(r)
print(f"Jarque-Bera (FE resid): stat={jb[0]:.2f}, p={jb[1]:.4f}")
print("Note: Durbin-Watson is invalid for panels; serial correlation handled via clustering/bootstrap.")

## 7. Environmental Kuznets Curve test *(R5)*

In [ ]:
e2=model_df.copy(); e2["gdp2"]=e2["gdp_per_capita"]**2; e2i=e2.set_index(["country","year"])
ekc=PanelOLS(e2i[YVAR],sm.add_constant(e2i[COV+["gdp2"]]),entity_effects=True).fit(cov_type="clustered",cluster_entity=True)
b1,b2=ekc.params["gdp_per_capita"],ekc.params["gdp2"]; tp=-b1/(2*b2) if b2 else np.nan
print(f"GDP p={ekc.pvalues['gdp_per_capita']:.3f} | GDP^2 coef={b2:.6f} p={ekc.pvalues['gdp2']:.3f}")
print(f"Turning point={tp:.1f} | GDP range={model_df['gdp_per_capita'].min():.0f}-{model_df['gdp_per_capita'].max():.0f}")
print("=> EKC supported ONLY if GDP^2 significant AND turning point inside range; otherwise report linear/negative only.")

## 8. Digitalization: narrowed interpretation *(R4)*

Internet penetration is a broad proxy for digital development, not a direct BDA measure. We
report within vs. between effects (Section 4), the coefficient CI, a **TOST equivalence** test,
and a **mediation check**. We do not assert "mediation" without a test.

In [ ]:
b=cfe.params["internet"]; se=cfe.std_errors["internet"]
print(f"Internet WITHIN (Country FE): {b:.5f}  95% CI [{b-1.96*se:.5f}, {b+1.96*se:.5f}]")
print(f"Internet BETWEEN (CRE m_internet): {mcre.params['m_internet']:.5f}  p={mcre.pvalues['m_internet']:.3f}")
bound=0.01; dfree=int(cfe.df_resid)
p_tost=max(1-stats.t.cdf((b+bound)/se,dfree), stats.t.cdf((b-bound)/se,dfree))
print(f"TOST equivalence (within, bound +-{bound}): p={p_tost:.3f} -> "
      f"{'within effect equivalent to zero' if p_tost<0.05 else 'equivalence not established'}")
# mediation check (gdp -> waste -> landfill), FE
am=PanelOLS(pdata['waste_per_capita'],sm.add_constant(pdata[['gdp_per_capita']]),entity_effects=True).fit(cov_type="clustered",cluster_entity=True)
bm=PanelOLS(y,sm.add_constant(pdata[['waste_per_capita','gdp_per_capita']]),entity_effects=True).fit(cov_type="clustered",cluster_entity=True)
print(f"Mediation check: a(gdp->waste) p={am.pvalues['gdp_per_capita']:.3f}, "
      f"b(waste->landfill|gdp) p={bm.pvalues['waste_per_capita']:.3f} -> no evidence of mediation.")

## 9. Fractional-response robustness *(R5)*

In [ ]:
cd2=pd.get_dummies(model_df["country"],prefix="c",drop_first=True).astype(float)
Xf=sm.add_constant(pd.concat([model_df[COV],cd2],axis=1).astype(float))
fl=sm.GLM(model_df[YVAR],Xf,family=sm.families.Binomial()).fit(cov_type="cluster",cov_kwds={"groups":model_df["country"]})
fit=fl.fittedvalues
print(f"Fractional logit fitted in [0,1]? min={fit.min():.3f} max={fit.max():.3f}")
frac=pd.DataFrame({"coef":fl.params[COV],"p":fl.pvalues[COV]}).round(4)
frac.to_csv(f"{OUTPUT_DIR}/T07_fractional_logit.csv"); print(frac.to_string())
print("Consistent with linear FE -> results not driven by functional form.")

## 10. Robustness: leave-one-country-out & exclude-interpolated *(R6)*

In [ ]:
loco=[]
for c in sorted(model_df["country"].unique()):
    s=model_df[model_df["country"]!=c].set_index(["country","year"])
    m=PanelOLS(s[YVAR],sm.add_constant(s[COV]),entity_effects=True).fit(cov_type="clustered",cluster_entity=True)
    loco.append({"dropped":c,**{v:round(m.params[v],5) for v in ["waste_per_capita","gdp_per_capita","internet"]}})
loco=pd.DataFrame(loco); loco.to_csv(f"{OUTPUT_DIR}/T08_leave_one_out.csv",index=False)
print("Leave-one-country-out (FE coefficients):"); print(loco.to_string(index=False))
mask=~(((model_df["country"].str.contains("Mal",case=False))&(model_df["year"]==2017))|
       ((model_df["country"].str.contains("Serb",case=False))&(model_df["year"]==2020)))
s=model_df[mask].set_index(["country","year"])
m=PanelOLS(s[YVAR],sm.add_constant(s[COV]),entity_effects=True).fit(cov_type="clustered",cluster_entity=True)
print(f"\nExcluding interpolated obs (N={int(m.nobs)}):")
print(pd.DataFrame({"coef":m.params[COV],"p":m.pvalues[COV]}).round(4).to_string())

## 11. Figures *(the submission contained none)*

In [ ]:
plt.rcParams.update({"font.size":11,"axes.spines.top":False,"axes.spines.right":False})
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(sel["K"],sel["inertia"],"o-"); ax[0].set_title("Inertia (elbow)"); ax[0].set_xlabel("K")
ax[1].plot(sel["K"],sel["silhouette"],"s-"); ax[1].set_title("Silhouette"); ax[1].set_xlabel("K")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/F1_cluster_selection.png",dpi=200,bbox_inches="tight"); plt.show()
pca=PCA(n_components=2).fit(Xz); PC=pca.transform(Xz)
plt.figure(figsize=(6.5,5))
for r in sorted(model_df["regime"].unique()):
    m=model_df["regime"].values==r; plt.scatter(PC[m,0],PC[m,1],label=f"Regime {r}",s=30)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.0f}%)"); plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.0f}%)")
plt.title("Predictor-side regimes (PCA)"); plt.legend()
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/F2_pca.png",dpi=200,bbox_inches="tight"); plt.show()
r=np.asarray(cfe.resids).ravel()
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].scatter(np.asarray(cfe.fitted_values).ravel(),r,s=18); ax[0].axhline(0,ls="--")
ax[0].set_title("Residuals vs fitted (FE)"); ax[0].set_xlabel("Fitted"); ax[0].set_ylabel("Residual")
qqplot(r,line="45",fit=True,ax=ax[1]); ax[1].set_title("Q-Q (FE residuals)")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/F3_residuals.png",dpi=200,bbox_inches="tight"); plt.show()
plt.figure(figsize=(9,5))
for c,g in df.groupby("country"): plt.plot(g["year"],g["landfill_rate"],marker="o",label=c,linewidth=1)
plt.ylabel("Landfill rate (0-1)"); plt.title("Landfill dependency by country, 2015-2023")
plt.legend(ncol=3,fontsize=8); plt.grid(alpha=.2)
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/F4_country_trajectories.png",dpi=200,bbox_inches="tight"); plt.show()

## 12. What to report

- **Main table:** Pooled OLS (benchmark) / Country FE / Two-way FE / RE, then **CRE (Mundlak)** as
  the main specification, with wild-bootstrap (Webb) CIs, N, G, R²(within/between/overall),
  AIC/BIC, and the Mundlak test.
- **Headline:** variation is overwhelmingly **between countries** (high between-share); **within
  countries** (2015–2023) there is **no robust association** with the covariates. Cross-country,
  landfill dependency is lower where waste generation and digitalization are higher (structural
  development gradient), reported cautiously given only 9 countries.
- **Digitalization:** no within-country effect (TOST); a between-country association only — a
  marker of structural development, not a direct operational lever. No mediation.
- **EKC:** report the quadratic test; drop EKC language unless supported.
- **Robustness:** fractional logit, leave-one-country-out, exclude-interpolated — all consistent.
- **Figures:** cluster selection, PCA, residual/Q-Q, country trajectories.


## 13. Download all outputs (tables + figures)

Bundles every CSV table and PNG figure in `outputs/` into (a) a single Excel workbook
(one sheet per table) and (b) a ZIP archive, and triggers a browser download in Colab.

In [ ]:
# --- Bundle & download all outputs ---
import os, glob, zipfile
try:
    import openpyxl  # noqa
except Exception:
    get_ipython().system('pip -q install openpyxl')

# (a) one Excel workbook, one sheet per CSV table
xlsx_path = f"{OUTPUT_DIR}/all_tables.xlsx"
csv_files = sorted(glob.glob(f"{OUTPUT_DIR}/*.csv"))
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xw:
    for f in csv_files:
        sheet = os.path.splitext(os.path.basename(f))[0][:31]
        pd.read_csv(f).to_excel(xw, sheet_name=sheet, index=False)
print(f"Excel workbook written: {xlsx_path}  ({len(csv_files)} sheets)")

# (b) ZIP with every CSV + PNG (+ the workbook)
zip_path = "landfill_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{OUTPUT_DIR}/*"):
        z.write(f, arcname=os.path.join("outputs", os.path.basename(f)))
print(f"ZIP archive written: {zip_path}")
print("Contents:")
for f in sorted(glob.glob(f"{OUTPUT_DIR}/*")):
    print("  ", os.path.basename(f))

# (c) trigger downloads in Colab (ignored outside Colab)
try:
    from google.colab import files
    files.download(zip_path)
    files.download(xlsx_path)
except Exception:
    print("\nNot on Colab - files are in the working directory and the outputs/ folder.")